# TradeSense · Component 03 · notebook 48b: control for notebook 48 — is the AE's gain just more alarms? (CPU, minutes, no training)

Notebook 48 showed that adding the AE to XGBoost-without-a-family raises macro recall from 0.193 to 0.473, but "either alarms" also roughly doubles the genuine alarm burden. This control gives XGBoost-without-a-family the same genuine burden (larger budget) and asks whether the AE still adds recall. It is a **new, labelled analysis** declared before it runs (text hashed below); notebook 48's results are not changed. The study models from notebook 48 are read from Drive; nothing is retrained. Run notebook 48 first.

In [ ]:
import os, sys, json, time, hashlib, warnings, glob, subprocess
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not in Colab -> using TRADESENSE_BASE.')
try:
    import xgboost as xgb
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'xgboost']); import xgboost as xgb
BASE = Path(os.environ.get('TRADESENSE_BASE', '/content/drive/MyDrive/research datasets'))
L3 = BASE/'layer3_v3'; OUT = BASE/'models_b2_v1'; XG = OUT/'xgb'; XG48 = OUT/'xgb_lofo_study'; XG48.mkdir(parents=True, exist_ok=True)
SYMBOLS = ['BTCUSDT', 'ETHUSDT']; SEEDS = [11, 22, 33]; MODE = 'N1'; CFG = ('A', 'pooled', 3)
MOCK = os.environ.get('TS_MOCK') == '1'; NBOOT = 200 if MOCK else 2000
MAX_RUN_MINUTES = float(os.environ.get('MAX_RUN_MINUTES', 150)); T_START = time.time(); NTHREAD = max(1, os.cpu_count() or 1)
LINES = []
def say(s=''): print(s); LINES.append(str(s))
def sha_file(p, n=16): return hashlib.sha256(Path(p).read_bytes()).hexdigest()[:n]
def out_of_time(): return (time.time() - T_START) / 60 > MAX_RUN_MINUTES
hf = L3/'protocol_bar_models_v2.sha256'; pf = L3/'protocol_bar_models_v2.md'
assert hf.exists() and pf.exists(), 'protocol B2 missing'
ph = hashlib.sha256(pf.read_text().encode()).hexdigest(); assert ph == hf.read_text().split()[0], 'frozen protocol B2 was changed'
pu = OUT/'protocol_u1_v2.md'; assert pu.exists(), 'run notebook 47 first (it hashes protocol U1 v2)'
PHU = hashlib.sha256(pu.read_text().encode()).hexdigest(); assert PHU.startswith('c448d8d48815aab0'), 'protocol U1 v2 differs from the approved text'
assert PHU == (OUT/'protocol_u1_v2.sha256').read_text().split()[0]
READ48 = r"""Protocol U1 v2, Part B - NEW labelled control analysis (notebook 48b), written BEFORE it is run. Not part of the approved protocol; it does not change any result of notebook 48.
Question: is the gain of 'S_wo or N' over S_wo caused by the AE, or only by the roughly doubled alarm budget?
1. Same study models (read from xgb_lofo_study, no training), same CAL-3 events, same T1c rule, same AE (declared q).
2. Control arm = S_wo alone at a larger budget b in {2, 3, 4, 5, 6} episodes/day (q chosen on the genuine DEV bars exactly as before). Matched budget b* = the smallest b whose mean realised genuine CAL episodes/day (over the six study models and both symbols) is >= the realised genuine CAL episodes/day of 'S_wo or N' in notebook 48 (decided on genuine bars only, no labels).
3. H-N1: macro recall(S_wo or N at 2/2) - macro recall(S_wo alone at b*) >= 0.05 with paired event-bootstrap 95% interval (2000 draws, seed 0, events resampled within symbol x family) above 0, on CAL-3. If it fails, the report says the gain is explained by alarm budget.
4. Descriptive: the whole S_wo curve (budget, realised genuine episodes/day, macro recall) and per-family recall at b*. VAL-3 is shown, not decided.
"""
RH = hashlib.sha256(READ48.encode()).hexdigest(); rp = OUT/'protocol_u1_v2_partB_control_readings.md'
if rp.exists(): assert rp.read_text() == READ48, 'operational readings exist and differ from this notebook'
else: rp.write_text(READ48); (OUT/'protocol_u1_v2_partB_control_readings.sha256').write_text(RH + '\n')
T32 = json.loads((OUT/'test_b_results_v1.json').read_text())
assert sha_file(OUT/'b2_lib.py') == T32['b2_lib_sha16'], 'b2_lib differs from the one that opened TEST'
for f_ in ['declared_ae_v2.json', 'xgb_selection_table.csv', 'xgb_val_table_T1c.csv']: assert (OUT/f_).exists(), f_ + ' missing'
DAE = json.loads((OUT/'declared_ae_v2.json').read_text()); assert DAE['variant'] == 'mean10' and DAE['normalisation'] == 'N1'
for s in SYMBOLS: assert (OUT/'scores'/f'ae_ctx_agg_{s}_N1.npz').exists() and (OUT/'scores'/f'ae_ctx_{s}_N1.npz').exists()
sys.path.insert(0, str(OUT)); import b2_lib as B
say(f'protocol U1 v2 sha256 {PHU[:16]}; control readings sha256 {RH[:16]}; protocol B2 intact {ph[:16]}; xgboost {xgb.__version__}; {NTHREAD} threads' + ('  [MOCK RUN]' if MOCK else ''))

Mounted at /content/drive
protocol U1 v2 sha256 c448d8d48815aab0; control readings sha256 0e13bd5ef3f6d8d6; protocol B2 intact 443f99d5faf5929f; xgboost 3.4.1; 2 threads


## 1 · Data, features, AE scores

In [ ]:
LOOK = 15; FEATS = B.FEATS; PB = B.PRIMARY; OLD = [1, 2, 3, 4, 5, 6]; AOLEN = 31 * 1440; CAL_LO = 1440
L1V = B.valid_mask(129600, 90720, 129600); CALV = B.valid_mask(AOLEN, CAL_LO, AOLEN)
def find1(pat):
    c = [x for x in glob.glob(str(BASE/'**'/pat), recursive=True) if 'backup' not in x.lower()]; assert c, pat; return sorted(c)[0]
D = {}; SCO = {}
for s in SYMBOLS:
    a = pd.read_parquet(find1(f'{s.lower()}_l1_feature_matrix_90d.parquet')); a = a.set_index('datetime') if 'datetime' in a.columns else a
    b = pd.read_parquet(BASE/'layer1_aug_oct_v1'/f'{s.lower()}_l1_feature_matrix_aug_oct_2024.parquet').iloc[:AOLEN]      # genuine August only: TEST-B is never loaded
    for d in (a, b): d.loc[d.index[:15], 'volume_acceleration'] = 0.0
    sets = {}
    for name in ['TRAIN', 'VAL', 'CAL']:
        f = pd.read_parquet(L3/f'l3v3_{s}_{name}.parquet'); ev = pd.read_csv(L3/f'l3v3_{s}_{name}_events.csv')
        sets[name] = {int(r): (g.reset_index(drop=True), ev[ev.replica == r].reset_index(drop=True)) for r, g in f.groupby('replica')}
    D[s] = {'GEN_L1': a[FEATS].reset_index(drop=True), 'GEN_AO': b[FEATS].reset_index(drop=True), 'sets': sets}
    z = np.load(OUT/'scores'/f'ae_ctx_{s}_{MODE}.npz'); SCO[(s, MODE)] = {k: z[k] for k in z.files if k != 'scale'}
    for k, v in SCO[(s, MODE)].items():
        n_ = len(D[s]['GEN_L1']) if k == 'GEN_L1' else len(D[s]['GEN_AO']) if k == 'GEN_AO' else len(D[s]['sets'][k.split('_r')[0]][int(k.split('_r')[1])][0])
        assert len(v) == n_ and v.shape[1] == 8, (s, k)
    say(f'{s}: genuine Layer 1 {len(a)} bars, genuine Aug {len(b)} bars; replicas ' + str({k: len(v) for k, v in sets.items()}))
assert not any('TEST' in k for s in SYMBOLS for k in SCO[(s, MODE)])
N0 = {s: B.fit_n0(D[s]['GEN_L1'].iloc[15:90720]) for s in SYMBOLS}

def build_feats(Z):
    """53 causal rolling features of the six normalised features (identical recipe to XGBoost v1/v2, look-back <= 15 bars)."""
    df = pd.DataFrame(np.asarray(Z, np.float64), columns=FEATS); cols = {}; prev15 = df.shift(1).rolling(15).mean()
    for f in FEATS:
        s_ = df[f]; a_ = s_.abs()
        cols[f + '__z'] = s_; cols[f + '__d1'] = s_ - s_.shift(1); cols[f + '__mean3'] = s_.rolling(3).mean(); cols[f + '__mean15'] = s_.rolling(15).mean(); cols[f + '__amax3'] = a_.rolling(3).max()
        cols[f + '__amax15'] = a_.rolling(15).max(); cols[f + '__std15'] = s_.rolling(15).std(); cols[f + '__dev'] = s_ - prev15[f]
    A = df.abs(); ne = (A > 3).sum(axis=1).astype(float); sa = A.sum(axis=1)
    cols['x__n_extreme'] = ne; cols['x__sumabs'] = sa; cols['x__n_extreme_mean5'] = ne.rolling(5).mean(); cols['x__sumabs_mean5'] = sa.rolling(5).mean(); cols['x__sumabs_max15'] = sa.rolling(15).max()
    out = pd.DataFrame(cols).to_numpy(np.float32, copy=True); out[:B.WARM] = np.nan; return out, list(cols)
NAMES_A = build_feats(np.zeros((200, 6)))[1]; NA = len(NAMES_A); assert NA == 53
XS = {}
def series_F(s, key):
    if key in ('GEN_L1', 'GEN_AO'): return D[s][key]
    name, r = key.split('_r'); return D[s]['sets'][name][int(r)][0]
def XA(s, key):
    """feature set A (53 columns) of a series under N1; CAL / genuine-August matrices are not cached."""
    k = (s, key)
    if k in XS: return XS[k]
    z = B.normalise(series_F(s, key), MODE, N0[s]); X = build_feats(z)[0]
    if not key.startswith(('CAL', 'GEN_AO')): XS[k] = X
    return X
RS = {}
for s in SYMBOLS:
    for name in ['TRAIN', 'VAL']:
        for r, (f, ev) in D[s]['sets'][name].items():
            lab = f.label.to_numpy().astype(int); inseg = f.in_seg.to_numpy().astype(bool); n = len(f); idx = np.arange(n)
            c = np.concatenate([[0], np.cumsum((lab != 0).astype(np.int64))])
            clean = ((c[idx + 1] - c[np.maximum(idx - LOOK, 0)]) == 0) & inseg & (idx >= B.WARM)
            pos = (lab > 0) & inseg & (idx >= B.WARM); assert not (clean & pos).any(); RS[(s, name, r)] = (lab, inseg, clean, pos)
# ---- events, AE scores (A6 = mean10, column 0 of the aggregated files)
def nanw(a): a = np.asarray(a, 'float64').copy(); a[:B.WARM] = np.nan; return a
ES = {}
for name in ['VAL', 'CAL']:
    ES[name] = [(s, f'{name}_r{r}', D[s]['sets'][name][r][1]) for s in SYMBOLS for r in sorted(D[s]['sets'][name])]
META = {n: pd.concat([pd.DataFrame({'sym': s, 'fam': ev.family.to_numpy()}) for s, k, ev in ES[n]], ignore_index=True) for n in ES}
for s, k, ev in ES['CAL']: assert ev.t0.min() >= CAL_LO and (ev.t0 + ev.dur).max() <= AOLEN, 'CAL-3 events must lie inside 2024-08-02..08-31'
for s, k, ev in ES['VAL']: assert ev.t0.min() >= 90720 and (ev.t0 + ev.dur).max() <= 129600
AE = {}
for s in SYMBOLS:
    z = np.load(OUT/'scores'/f'ae_ctx_agg_{s}_N1.npz'); assert not any('TEST' in k for k in z.files)
    for k in z.files:
        if k == 'scale' or k.startswith('TRAIN'): continue
        a = z[k][:, 0]; a = a[:AOLEN] if k.startswith('CAL') else a; AE[(s, k)] = nanw(a)
say('events per set: ' + str({n: META[n].groupby('sym').size().to_dict() for n in META}) + '; CAL-3 per family (both symbols): ' + str(META['CAL'].fam.value_counts().sort_index().to_dict()))
# ---- the selection recorded by notebook 29 (must be the config this protocol studies)
TAB = pd.read_csv(OUT/'xgb_selection_table.csv'); tm = TAB[TAB['mode'] == MODE]
bestA = tm[tm.fs == 'A'].metric.max(); bestB = tm[tm.fs == 'B'].metric.max(); al_ = 'B' if bestB - bestA >= 0.03 else 'A'; tt = tm[tm.fs == al_]; tied = tt[tt.metric >= tt.metric.max() - 0.05].copy()
tied['k'] = list(zip((tied.scope != 'pooled').astype(int), tied.depth, -tied.pr_auc)); FROZEN = tied.sort_values('k').iloc[0].config
if not MOCK: assert FROZEN == 'A|pooled|d3', f'frozen N1 selection is {FROZEN}, protocol U1 studies A|pooled|d3'
else: say(f'[MOCK] frozen selection in the mock is {FROZEN}; the study still uses A|pooled|d3')
say('data ready; XGBoost config studied: A | pooled | depth 3, N1; frozen selection recorded by notebook 29: ' + FROZEN)

BTCUSDT: genuine Layer 1 129600 bars, genuine Aug 44640 bars; replicas {'TRAIN': 2, 'VAL': 2, 'CAL': 1}
ETHUSDT: genuine Layer 1 129600 bars, genuine Aug 44640 bars; replicas {'TRAIN': 2, 'VAL': 2, 'CAL': 1}
events per set: {'VAL': {'BTCUSDT': 300, 'ETHUSDT': 300}, 'CAL': {'BTCUSDT': 200, 'ETHUSDT': 200}}; CAL-3 per family (both symbols): {1: 50, 2: 50, 3: 50, 4: 50, 5: 50, 6: 50, 7: 50, 8: 50}
data ready; XGBoost config studied: A | pooled | depth 3, N1; frozen selection recorded by notebook 29: A|pooled|d3


## 2 · Study models: XGBoost without one family (18 models)

In [ ]:
PARAMS = {'objective': 'binary:logistic', 'eval_metric': 'aucpr', 'tree_method': 'hist', 'eta': 0.05, 'subsample': 0.8, 'colsample_bytree': 0.8, 'min_child_weight': 5, 'reg_lambda': 1.0, 'scale_pos_weight': 5.0, 'nthread': NTHREAD}
MAX_ROUNDS, ES_ROUNDS = 800, 40
ROWLOG = []
def get_rows(s, name, lofo, tag):
    """positives from every replica; clean negatives from replica 0 only. VAL: only the 6 old families; the held-out family removed from both."""
    Xs, ys = [], []
    for r, (f, ev) in D[s]['sets'][name].items():
        lab, inseg, clean, pos = RS[(s, name, r)]; p = pos.copy()
        if name == 'VAL': p &= (lab <= 6)
        p &= (lab != lofo)
        m = p | (clean if r == 0 else np.zeros(len(lab), bool)); ii = np.flatnonzero(m)
        ROWLOG.append({'tag': tag, 'sym': s, 'set': name, 'min_bar': int(ii.min()), 'max_bar': int(ii.max()), 'n': int(len(ii)), 'n_pos': int(p[ii].sum()), 'pos_labels': sorted(set(lab[ii][p[ii]].tolist())), 'outside_segment': int((~inseg[ii]).sum())})
        Xs.append(XA(s, f'{name}_r{r}')[ii]); ys.append(p[ii].astype(np.float32))
    return np.concatenate(Xs), np.concatenate(ys)
def spath(fam, seed): return XG48/f'N1_A_pooled_pool_d3_s{seed}_no{fam}.json'
def train_study(fam, seed):
    p = spath(fam, seed); ip = Path(str(p)[:-5] + '.info.json')
    if p.exists() and ip.exists(): return False
    tag = f'study|no{fam}|s{seed}'; n0_ = len(ROWLOG)
    parts = [get_rows(s, 'TRAIN', fam, tag + '|fit') for s in SYMBOLS]; ves = [get_rows(s, 'VAL', fam, tag + '|es') for s in SYMBOLS]
    Xt = np.concatenate([x[0] for x in parts]); yt = np.concatenate([x[1] for x in parts]); Xv = np.concatenate([x[0] for x in ves]); yv = np.concatenate([x[1] for x in ves])
    dtr = xgb.DMatrix(Xt, label=yt, feature_names=NAMES_A); dva = xgb.DMatrix(Xv, label=yv, feature_names=NAMES_A); t = time.time()
    bst = xgb.train({**PARAMS, 'max_depth': 3, 'seed': seed}, dtr, num_boost_round=MAX_ROUNDS, evals=[(dva, 'val')], early_stopping_rounds=ES_ROUNDS, verbose_eval=False)
    bst.save_model(str(p)); info = {'family_removed': fam, 'best_iteration': int(bst.best_iteration), 'best_aucpr_val': float(bst.best_score), 'n_fit': int(len(yt)), 'n_fit_pos': int(yt.sum()), 'n_es': int(len(yv)), 'n_es_pos': int(yv.sum()), 'seconds': round(time.time() - t, 1), 'rows': ROWLOG[n0_:]}
    ip.write_text(json.dumps(info)); say(f'  trained study model without family {fam} ({B.FAMS[fam]}), seed {seed}: best iteration {info["best_iteration"]}, fit {len(yt):,} rows ({int(yt.sum()):,} positive), {info["seconds"]}s'); return True
class SM:
    """mean-margin ensemble of boosters from a list of files"""
    def __init__(self, paths): self.bst = [(lambda b_, p: (b_.load_model(str(p)), b_)[1])(xgb.Booster(), p) for p in paths]; self.its = [json.loads(Path(str(p)[:-5] + '.info.json').read_text())['best_iteration'] for p in paths]; self.cache = {}
    def score(self, s, key):
        if (s, key) not in self.cache:
            dm = xgb.DMatrix(XA(s, key), feature_names=NAMES_A); m = np.mean([b_.predict(dm, output_margin=True, iteration_range=(0, bi + 1)) for b_, bi in zip(self.bst, self.its)], axis=0).astype(np.float32).astype('float64'); m[:B.WARM] = np.nan      # float32 as in notebook 29b
            if key.startswith('CAL'): m = m[:AOLEN]
            self.cache[(s, key)] = m
        return self.cache[(s, key)]
TRAIN_DONE = True
say('--- training 6 families x 3 seeds of study models (A | pooled | depth 3, N1) ---')
for fam in OLD:
    for sd in SEEDS:
        if out_of_time(): TRAIN_DONE = False; break
        train_study(fam, sd)
if not TRAIN_DONE: say('>>> time budget reached: Run all again to continue (finished models are kept); the evaluation below is skipped until all 18 models exist. <<<')
else:
    ok = True
    for p in XG48.glob('*.info.json'):
        inf = json.loads(p.read_text())
        for r in inf['rows']:
            ok &= r['outside_segment'] == 0 and inf['family_removed'] not in r['pos_labels'] and not any(l > 6 for l in r['pos_labels'])
            if r['tag'].endswith('|fit'): ok &= r['max_bar'] < 90720 and r['min_bar'] >= B.WARM
            if r['tag'].endswith('|es'): ok &= r['min_bar'] >= 90720 and r['max_bar'] < 129600
    say(('PASS' if ok else 'FAIL') + ' T01 fit rows inside TRAIN-3, early-stopping rows inside VAL-3, held-out family and the two new families never used'); assert ok

--- training 6 families x 3 seeds of study models (A | pooled | depth 3, N1) ---
PASS T01 fit rows inside TRAIN-3, early-stopping rows inside VAL-3, held-out family and the two new families never used


## 3 · Evaluation helpers and tests

In [ ]:
def event_hits(alarm, ev):
    c = np.concatenate([[0], np.cumsum(alarm.astype(np.int64))]); t0 = ev.t0.to_numpy(); du = ev.dur.to_numpy(); return (c[t0 + du] - c[t0]) > 0
def rolling_inf(g, q): t = B.rolling_thr(g, q); return np.where(np.isnan(t), np.inf, t)
class Scorer:
    """T1c alarms of a score function fn(sym, key) -> float64 series. q from the genuine DEV bars at 2 episodes/day, per symbol."""
    def __init__(self, fn, bud=PB, qfix=None):
        self.fn = fn; self.q = {}; self.thr = {}
        for s in SYMBOLS:
            g = fn(s, 'GEN_L1'); self.q[s] = B.t1_q(g, L1V, bud) if qfix is None else qfix[s]; self.thr[(s, 'L1')] = rolling_inf(g, self.q[s]); self.thr[(s, 'AO')] = rolling_inf(fn(s, 'GEN_AO'), self.q[s])
    def event_alarms(self, name):
        ref = 'L1' if name == 'VAL' else 'AO'; return [self.fn(s, k) > self.thr[(s, ref)] for s, k, ev in ES[name]]
    def genuine(self, where):
        return {s: (self.fn(s, 'GEN_L1') > self.thr[(s, 'L1')]) if where == 'DEV' else (self.fn(s, 'GEN_AO') > self.thr[(s, 'AO')]) for s in SYMBOLS}
def hits_of(alarms, name): return np.concatenate([event_hits(a, ev) for a, (s, k, ev) in zip(alarms, ES[name])])
def eps_of(alarm_by_sym, where): valid = L1V if where == 'DEV' else CALV; return float(np.mean([B.eps_per_day(alarm_by_sym[s], valid) for s in SYMBOLS]))
def fam_rate(h, name, fam):
    m = META[name]; return float(np.mean([h[((m.sym == s) & (m.fam == fam)).to_numpy()].mean() for s in SYMBOLS]))
def pair_boot(hA, hB, name, fams, n=None, seed=0):
    n = n or NBOOT; m = META[name]; groups = []
    for s in SYMBOLS:
        gs = [np.flatnonzero(((m.sym == s) & (m.fam == f)).to_numpy()) for f in fams]; groups.append([g for g in gs if len(g)])
    d = hA.astype(float) - hB.astype(float); rg = np.random.default_rng(seed); out = []
    for _ in range(n): out.append(np.mean([np.mean([d[g[rg.integers(0, len(g), len(g))]].mean() for g in gs]) for gs in groups]))
    return tuple(float(x) for x in np.percentile(out, [2.5, 97.5]))
# ---- unit tests of the helpers (do not need the study models)
rg = np.random.default_rng(0); _, _, ev_ = ES['CAL'][0]
for _ in range(3):
    al = rg.random(AOLEN) < 0.002; assert (event_hits(al, ev_) == np.array([al[r.t0:r.t0 + r.dur].any() for r in ev_.itertuples()])).all()
say('PASS T02 event hit rule equals a plain loop')
hh = rg.random(len(META['CAL'])) < 0.5; lo_, hi_ = pair_boot(hh, hh, 'CAL', OLD, n=50); assert lo_ == 0 and hi_ == 0; say('PASS T03 paired bootstrap of identical arms is exactly 0')
AEs = Scorer(lambda s, k: AE[(s, k)], qfix=DAE['q_T1_by_budget']['2'])
RQ = {s: B.t1_q(AE[(s, 'GEN_L1')], L1V, PB) for s in SYMBOLS}
say(('PASS' if all(abs(RQ[s] - AEs.q[s]) < 1e-12 for s in SYMBOLS) else 'INFO') + ' T04 AE alarm lines: declared q (used) ' + str(AEs.q) + '; recomputed from the saved scores ' + str(RQ) + ('' if all(abs(RQ[s] - AEs.q[s]) < 1e-12 for s in SYMBOLS) else ' (saved scores are float32, borderline episode counts can move one grid step; the declared q is the deployed one)'))

PASS T02 event hit rule equals a plain loop
PASS T03 paired bootstrap of identical arms is exactly 0
PASS T04 AE alarm lines: declared q (used) {'BTCUSDT': 0.993, 'ETHUSDT': 0.99}; recomputed from the saved scores {'BTCUSDT': 0.993, 'ETHUSDT': 0.99}


## 4 · Control results

In [ ]:
RESULT = {}
BUDS = [2, 3, 4, 5, 6]
if not TRAIN_DONE: say('study models incomplete -> run notebook 48 to the end first')
else:
    S_AL = {}; S_G = {}
    models = {fam: SM([spath(fam, sd) for sd in SEEDS]) for fam in OLD}
    for fam in OLD:
        for b in BUDS:
            Ss = Scorer(models[fam].score, bud=b); S_AL[(fam, b)] = {n: Ss.event_alarms(n) for n in ['VAL', 'CAL']}; S_G[(fam, b)] = eps_of(Ss.genuine('CAL'), 'CAL')
    def hits_f(alarm_lists, name, fam): h = hits_of(alarm_lists, name); sel = (META[name].fam == fam).to_numpy(); out = np.zeros(len(h), bool); out[sel] = h[sel]; return out
    def macro_h(h, name): return float(np.mean([np.mean([h[((META[name].sym == s) & (META[name].fam == f)).to_numpy()].mean() for f in OLD]) for s in SYMBOLS]))
    HS = {}; HOR = {}; ONG = {}
    for name in ['VAL', 'CAL']:
        HOR[name] = np.zeros(len(META[name]), bool); AEa = AEs.event_alarms(name)
        for fam in OLD:
            S2 = S_AL[(fam, 2)][name]; hs = hits_f([x | y for x, y in zip(S2, AEa)], name, fam); HOR[name] |= hs
            for b in BUDS: HS[(name, b)] = HS.get((name, b), np.zeros(len(META[name]), bool)) | hits_f(S_AL[(fam, b)][name], name, fam)
    # genuine burden of 'S_wo or N' (budget 2 + AE), CAL
    or_g = float(np.mean([eps_of({s: Scorer(models[fam].score, bud=2).genuine('CAL')[s] | AEs.genuine('CAL')[s] for s in SYMBOLS}, 'CAL') for fam in OLD]))
    r48 = json.loads((OUT/'u1_partB_results.json').read_text()); say(('PASS' if abs(r48['H-U5']['OR_mean_eps_day'] - or_g) < 1e-9 else 'FAIL') + f' T08 genuine burden of "S_wo or N" equals notebook 48 ({or_g:.3f} vs {r48["H-U5"]["OR_mean_eps_day"]:.3f})'); assert abs(r48['H-U5']['OR_mean_eps_day'] - or_g) < 1e-9 or r48.get('mock')
    CURVE = []
    for b in BUDS:
        CURVE.append({'budget': b, 'S_wo genuine CAL eps/day': float(np.mean([S_G[(f, b)] for f in OLD])), 'S_wo macro recall VAL': macro_h(HS[('VAL', b)], 'VAL'), 'S_wo macro recall CAL': macro_h(HS[('CAL', b)], 'CAL')})
    CV = pd.DataFrame(CURVE); CV.to_csv(OUT/'u1_partB_control_curve.csv', index=False)
    bstar = next((b for b in BUDS if CV.loc[CV.budget == b, 'S_wo genuine CAL eps/day'].iloc[0] >= or_g), BUDS[-1]); matched = bool(CV.loc[CV.budget == bstar, 'S_wo genuine CAL eps/day'].iloc[0] >= or_g)
    say(f'genuine CAL episodes/day of "S_wo or N" (notebook 48): {or_g:.2f}')
    say('--- XGBoost-without-the-family alone at larger budgets (mean over the six study models) ---'); say(CV.round(3).to_string(index=False))
    say(f'matched budget b* = {bstar} episodes/day' + ('' if matched else ' (even the largest budget does not reach the burden of "S_wo or N"; control is conservative against the AE)'))
    rec_or = macro_h(HOR['CAL'], 'CAL'); rec_s = macro_h(HS[('CAL', bstar)], 'CAL'); lo, hi = pair_boot(HOR['CAL'], HS[('CAL', bstar)], 'CAL', OLD)
    rows = []
    for fam in OLD:
        sel = (META['CAL'].fam == fam).to_numpy(); rows.append({'family': B.FAMS[fam], 'S_wo at 2': fam_rate(HS[('CAL', 2)], 'CAL', fam), f'S_wo at b*={bstar}': fam_rate(HS[('CAL', bstar)], 'CAL', fam), 'S_wo or N (2/2)': fam_rate(HOR['CAL'], 'CAL', fam)})
    FT2 = pd.DataFrame(rows); FT2.to_csv(OUT/'u1_partB_control_family_table.csv', index=False); say('--- CAL-3 per family ---'); say(FT2.round(3).to_string(index=False))
    hn1 = bool(rec_or - rec_s >= 0.05 and lo > 0)
    RESULT = {'control_readings_sha256': RH, 'genuine_cal_eps_day_or': or_g, 'b_star': bstar, 'matched': matched, 'recall_or': rec_or, 'recall_s_at_bstar': rec_s, 'diff': rec_or - rec_s, 'ci': [lo, hi], 'H-N1': hn1, 'mock': MOCK}
    say(''); say(f'H-N1 (new control): recall(S_wo or N) {rec_or:.3f} - recall(S_wo alone at {bstar}/day, same genuine burden) {rec_s:.3f} = {rec_or - rec_s:+.3f}, paired 95% {lo:+.3f}..{hi:+.3f} -> {"HOLDS: the AE adds more than extra alarms would" if hn1 else "FAILS: the gain is explained by the larger alarm budget"}')
    say('New labelled analysis; notebook 48 results unchanged. Labels are synthetic.')

PASS T08 genuine burden of "S_wo or N" equals notebook 48 (3.486 vs 3.486)
genuine CAL episodes/day of "S_wo or N" (notebook 48): 3.49
--- XGBoost-without-the-family alone at larger budgets (mean over the six study models) ---
 budget  S_wo genuine CAL eps/day  S_wo macro recall VAL  S_wo macro recall CAL
      2                     1.506                  0.221                  0.193
      3                     2.000                  0.241                  0.210
      4                     2.756                  0.272                  0.257
      5                     3.850                  0.316                  0.303
      6                     4.161                  0.325                  0.317
matched budget b* = 5 episodes/day
--- CAL-3 per family ---
                   family  S_wo at 2  S_wo at b*=5  S_wo or N (2/2)
       pump_and_dump_like       0.44          0.60             0.64
       fake_breakout_like       0.36          0.56             0.46
    abnormal_volume_burst    

## 5 · Save

In [ ]:
if RESULT:
    (OUT/'u1_partB_control_results.json').write_text(json.dumps(RESULT, indent=1, default=str)); (OUT/'u1_partB_control_share_block.txt').write_text('\n'.join(LINES))
    say('Saved: u1_partB_control_results.json, u1_partB_control_curve.csv, u1_partB_control_family_table.csv, u1_partB_control_share_block.txt in models_b2_v1/ . Send the share block to Claude.')
else: say('Nothing saved (study models missing).')

Saved: u1_partB_control_results.json, u1_partB_control_curve.csv, u1_partB_control_family_table.csv, u1_partB_control_share_block.txt in models_b2_v1/ . Send the share block to Claude.
